# QUANTUM LAB — Oscilador armónico cuántico 1D

Polinomios de Hermite, niveles equiespaciados Eₙ = ħω(n + ½), energía de punto cero, zona clásicamente prohibida, teorema del virial, límite clásico y dinámica de superposiciones.

**Criterio del curso que cumple este cuaderno**

- *Función de onda, densidad de probabilidad, densidad de energía cinética, densidad de energía potencial y energía frente
  a la posición, para cualquier estado del oscilador.* Se escribe el número cuántico **n** (y la masa y ω); la pestaña
  **«Estado individual»** dibuja ψₙ(x), |ψₙ(x)|², V(x) con el nivel Eₙ, u_T(x) = (ħ²/2m)|ψ'|² y u_V(x) = V|ψ|².

**Cómo usarlo en Google Colab**

1. Menú **Entorno de ejecución → Ejecutar todo** (o pulsa ▶ en cada celda, en orden). No hace falta instalar nada.
2. Al final aparece el simulador: los **parámetros** arriba y los resultados organizados en **pestañas**.
3. Escribe **cualquier valor** en las casillas y pulsa *Enter* (o mueve el deslizador, que amplía su rango solo).
   Se admiten formas como `2.5`, `2,5`, `1e-30`, `3*10^8` o `2*pi`.
4. Si un valor no tiene sentido físico (masa negativa, n = 2.5, una caja más pequeña que la longitud de Planck…),
   el simulador **no se detiene**: el apartado **«VALIDEZ FÍSICA DE LOS DATOS»** explica qué ocurre y por qué.
   - **[ERROR]**: con ese dato no se puede calcular (se explica el motivo físico).
   - **[AVISO]**: se calcula, pero el resultado no tiene respaldo físico (por ejemplo, el modelo deja de valer).
   - **[NOTA]**: información útil para interpretar el resultado.
5. Cada resultado se presenta como en los simuladores de Bohr y del efecto fotoeléctrico: parámetros, validez física,
   resultados alineados, tablas, interpretación física y gráficas.

Para ver el código de una celda, haz doble clic sobre su título.


In [ ]:
#@title 1) Utilidades comunes: formato de resultados, entradas sin límite y validez física { display-mode: "form" }
# =====================================================================================
#  Esta celda no contiene física: solo da formato a los resultados (recuadros, columnas
#  alineadas, tablas), lee los valores escritos por el usuario y explica los datos que
#  no tienen sentido físico. Ejecútala antes que las demás.
# =====================================================================================
import ast
import math
import operator
import textwrap
import warnings
import html as _html

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as w
from IPython.display import display, clear_output, HTML
from scipy import constants as const

warnings.filterwarnings("ignore", message=".*IPython.display.IFrame.*")
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ------------------------------------------------------------------
# Constantes físicas (CODATA, vía scipy.constants)
# ------------------------------------------------------------------
HBAR = const.hbar                    # J·s
H_PLANCK = const.h                   # J·s
C_LUZ = const.c                      # m/s
M_E = const.m_e                      # kg
EV = const.e                         # J por eV
K_B = const.k                        # J/K
A0 = const.physical_constants["Bohr radius"][0]          # m
E_H = const.physical_constants["Hartree energy"][0]      # J
L_PLANCK = const.physical_constants["Planck length"][0]  # m
KT_AMBIENTE_EV = K_B * 300 / EV      # energía térmica a 300 K  ≈ 0.0259 eV
ME_C2_EV = M_E * C_LUZ ** 2 / EV     # energía en reposo del electrón ≈ 511 keV

# ------------------------------------------------------------------
# Formato numérico
# ------------------------------------------------------------------
ANCHO = 80


def num(x, cifras=6):
    """Número legible: decimal si es «normal», notación científica si es muy grande o muy pequeño."""
    if x is None:
        return "—"
    try:
        x = float(x)
    except (TypeError, ValueError):
        return str(x)
    if math.isnan(x):
        return "no definido"
    if math.isinf(x):
        return "infinito" if x > 0 else "−infinito"
    if x == 0:
        return "0"
    if 1e-3 <= abs(x) < 1e6:
        texto = f"{x:.{cifras}g}"
    else:
        texto = f"{x:.{max(cifras - 2, 2)}e}"
    return texto.replace("-", "−")


def num_log(log10_valor, cifras=4):
    """Escribe 10^(log10_valor) aunque el número no quepa en un float (por ejemplo 10^5000)."""
    if not np.isfinite(log10_valor):
        return "0" if log10_valor < 0 else "infinito"
    exponente = math.floor(log10_valor)
    mantisa = 10 ** (log10_valor - exponente)
    if mantisa >= 9.9999999:
        mantisa, exponente = 1.0, exponente + 1
    if -3 <= exponente < 6:
        return num(10 ** log10_valor, cifras + 2)
    return f"{mantisa:.{cifras}f}e{exponente:+03d}".replace("-", "−")


def entero(n):
    """Entero con separador de miles (espacio fino) para leer números enormes."""
    return f"{int(n):,}".replace(",", " ")


_ESCALAS_LONGITUD = [(1e-15, "fm"), (1e-12, "pm"), (1e-9, "nm"), (1e-6, "µm"), (1e-3, "mm"), (1.0, "m"), (1e3, "km")]


def escala_longitud(L_m):
    """Unidad cómoda para dibujar una longitud L (en metros): devuelve (factor, nombre)."""
    for factor, nombre in reversed(_ESCALAS_LONGITUD):
        if L_m >= 0.1 * factor:
            return factor, nombre
    return 1e-15, "fm"


# ------------------------------------------------------------------
# Impresión con el estilo de QUANTUM LAB (recuadros y columnas alineadas)
# ------------------------------------------------------------------
def titulo_caja(titulo):
    print()
    print("╔" + "═" * (ANCHO - 2) + "╗")
    print("║" + titulo.center(ANCHO - 2) + "║")
    print("╚" + "═" * (ANCHO - 2) + "╝")


def seccion(titulo):
    print()
    print("─" * ANCHO)
    print(titulo)
    print("─" * ANCHO)


def dato(etiqueta, valor, unidad="", ancho_etiqueta=40):
    texto = f"{valor} {unidad}".rstrip()
    print(f"   {etiqueta:<{ancho_etiqueta}}: {texto}")


def parrafo(texto, sangria=3):
    for linea in textwrap.wrap(texto, ANCHO - sangria):
        print(" " * sangria + linea)


def vinetas(lineas, sangria=3):
    for linea in lineas:
        partes = textwrap.wrap(linea, ANCHO - sangria - 2)
        for i, parte in enumerate(partes):
            print(" " * sangria + ("• " if i == 0 else "  ") + parte)


def tabla(encabezados, filas, sangria=5, alinear=None):
    """Tabla con bordes de caja; cada columna se ajusta a su contenido. alinear: «c» centrado, «l» izquierda."""
    filas = [[str(c) for c in f] for f in filas]
    anchos = [max(len(str(h)), *(len(f[i]) for f in filas)) + 4 if filas else len(str(h)) + 4
              for i, h in enumerate(encabezados)]
    margen = " " * sangria

    def linea(izq, cruce, der):
        return margen + izq + cruce.join("─" * a for a in anchos) + der

    print(linea("┌", "┬", "┐"))
    alinear = alinear or "c" * len(encabezados)

    def celda(c, a, modo):
        return (" " + c).ljust(a) if modo == "l" else c.center(a)

    print(margen + "│" + "│".join(str(h).center(a) for h, a in zip(encabezados, anchos)) + "│")
    print(linea("├", "┼", "┤"))
    for f in filas:
        print(margen + "│" + "│".join(celda(c, a, m) for c, a, m in zip(f, anchos, alinear)) + "│")
    print(linea("└", "┴", "┘"))


# ------------------------------------------------------------------
# Validez física de los datos
# ------------------------------------------------------------------
class Diagnostico:
    """
    Reúne los comentarios sobre los datos introducidos:
      ERROR  no se puede calcular (por ejemplo, una masa negativa o n = 2.5);
      AVISO  se calcula, pero el resultado no tiene respaldo físico o el modelo deja de valer;
      NOTA   información útil para interpretar los resultados.
    """

    def __init__(self):
        self.items = []

    def error(self, texto):
        self.items.append(("ERROR", texto))

    def aviso(self, texto):
        self.items.append(("AVISO", texto))

    def nota(self, texto):
        self.items.append(("NOTA", texto))

    @property
    def hay_error(self):
        return any(nivel == "ERROR" for nivel, _ in self.items)

    def imprimir(self):
        titulo_caja("VALIDEZ FÍSICA DE LOS DATOS")
        if not self.items:
            print("   Todos los valores están dentro del rango de validez del modelo.")
            return
        orden = {"ERROR": 0, "AVISO": 1, "NOTA": 2}
        for nivel, texto in sorted(self.items, key=lambda it: orden[it[0]]):
            etiqueta = f"   [{nivel}] "
            partes = textwrap.wrap(texto, ANCHO - len(etiqueta))
            for i, parte in enumerate(partes):
                print((etiqueta if i == 0 else " " * len(etiqueta)) + parte)
            print()
        if self.hay_error:
            print("   >> Con estos datos no se puede calcular. Corrige los valores marcados como ERROR.")


def revisar_masa(m_me, diag, nombre="La masa"):
    """Comprueba una masa expresada en masas del electrón. Devuelve la masa en kg o None."""
    if m_me is None:
        return None
    if m_me == 0:
        diag.error(f"{nombre} vale 0. Una partícula sin masa (como el fotón) no obedece la ecuación de "
                   "Schrödinger: siempre viaja a la velocidad de la luz y necesita una descripción relativista. "
                   "Además, todas las energías del modelo serían infinitas (E ∝ 1/m).")
        return None
    if m_me < 0:
        diag.error(f"{nombre} es negativa ({num(m_me)} mₑ). No existe materia con masa negativa: la energía "
                   "cinética p²/2m saldría negativa y los estados ligados no tendrían sentido.")
        return None
    if m_me < 1e-6:
        diag.aviso(f"{nombre} ({num(m_me)} mₑ = {num(m_me * ME_C2_EV)} eV/c²) es menor que la de cualquier "
                   "partícula con masa conocida (los neutrinos tienen menos de ~1e−6 mₑ). Se calcula, pero no "
                   "corresponde a ninguna partícula real.")
    elif m_me > 1e15:
        diag.nota(f"{nombre} ({num(m_me * M_E)} kg) es la de un objeto macroscópico o casi. Su longitud de onda "
                  "de De Broglie es diminuta y los efectos cuánticos son imperceptibles: se comporta de forma "
                  "clásica (principio de correspondencia).")
    return m_me * M_E


def revisar_longitud(L_m, diag, nombre, que="longitud"):
    """Comentarios sobre una longitud física (en metros) ya validada como positiva."""
    if L_m < L_PLANCK:
        diag.aviso(f"{nombre} ({num(L_m)} m) es menor que la longitud de Planck (1.6e−35 m). A esa escala la "
                   "gravedad cuántica es importante y ninguna teoría física conocida es fiable: el resultado es "
                   "solo la evaluación matemática de la fórmula.")
    elif L_m < 1e-15:
        diag.aviso(f"{nombre} ({num(L_m)} m) es menor que un protón (≈ 1e−15 m). La partícula estaría confinada "
                   "en una región más pequeña que un núcleo: allí dominan la física nuclear y la relativista.")
    elif L_m > 1.0:
        diag.nota(f"{nombre} ({num(L_m)} m) es de tamaño macroscópico. Los niveles de energía quedan tan juntos que "
                  "el espectro es prácticamente continuo: es el límite clásico.")


def revisar_relatividad(log10_E_J, m_kg, diag, que="La energía"):
    """Compara una energía con la energía en reposo mc² (en escala logarítmica para no desbordar)."""
    log10_mc2 = math.log10(m_kg * C_LUZ ** 2)
    cociente = log10_E_J - log10_mc2
    if cociente > 0:
        diag.aviso(f"{que} es {num_log(cociente, 3)} veces la energía en reposo mc² de la partícula. Con tanta "
                   "energía se crearían pares partícula–antipartícula y la ecuación de Schrödinger (no relativista) "
                   "deja de valer: los números son el resultado matemático de la fórmula, no una predicción física.")
    elif cociente > -2:
        diag.aviso(f"{que} es el {num(100 * 10 ** cociente, 3)} % de mc²: la velocidad de la partícula es una "
                   "fracción apreciable de la de la luz y el modelo no relativista empieza a fallar.")


# ------------------------------------------------------------------
# Lectura segura de los valores escritos (admite 1e-30, 2,5, 2*pi, 3×10^8…)
# ------------------------------------------------------------------
_OPERACIONES = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
                ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}
_NOMBRES = {"pi": math.pi, "e": math.e, "inf": math.inf, "infinito": math.inf}


def _evaluar(nodo):
    if isinstance(nodo, ast.Expression):
        return _evaluar(nodo.body)
    if isinstance(nodo, ast.Constant) and isinstance(nodo.value, (int, float)):
        return float(nodo.value)
    if isinstance(nodo, ast.Name) and nodo.id.lower() in _NOMBRES:
        return _NOMBRES[nodo.id.lower()]
    if isinstance(nodo, ast.BinOp) and type(nodo.op) in _OPERACIONES:
        a, b = _evaluar(nodo.left), _evaluar(nodo.right)
        if isinstance(nodo.op, ast.Pow) and abs(b) > 1e4 and abs(a) > 1:
            return math.inf if b > 0 else 0.0
        return _OPERACIONES[type(nodo.op)](a, b)
    if isinstance(nodo, ast.UnaryOp) and type(nodo.op) in _OPERACIONES:
        return _OPERACIONES[type(nodo.op)](_evaluar(nodo.operand))
    raise ValueError


def leer_numero(texto):
    """Convierte el texto escrito en un número. Devuelve (valor, None) o (None, explicación)."""
    t = str(texto).strip()
    if t == "":
        return None, "la casilla está vacía"
    t = (t.replace("π", "pi").replace("−", "-").replace("×", "*").replace("^", "**").replace(" ", ""))
    if t.count(",") == 1 and "." not in t:
        t = t.replace(",", ".")                          # coma decimal: 2,5 → 2.5
    t = t.replace("x10", "*10").replace("X10", "*10")
    try:
        valor = _evaluar(ast.parse(t, mode="eval"))
    except OverflowError:
        return None, (f"«{texto}» es mayor que 1.8e308, el número más grande que puede manejar el ordenador "
                      "(un float de 64 bits)")
    except ZeroDivisionError:
        return None, f"«{texto}» contiene una división entre cero"
    except (ValueError, SyntaxError, TypeError, RecursionError, MemoryError):
        return None, f"«{texto}» no es un número (ejemplos válidos: 2.5, 2,5, 1e-30, 3*10^8, 2*pi)"
    if isinstance(valor, complex):
        return None, f"«{texto}» da un número complejo"
    if math.isinf(valor) and "inf" not in t.lower():
        return None, (f"«{texto}» es mayor que 1.8e308, el número más grande que puede manejar el ordenador "
                      "(un float de 64 bits)")
    return float(valor), None


# ------------------------------------------------------------------
# Entradas: deslizador + casilla de texto, SIN límite de rango
# ------------------------------------------------------------------
_ESTILO = {"description_width": "0px"}


class Entrada:
    """
    Control con deslizador y casilla de texto enlazados. La casilla admite CUALQUIER valor: si se escribe
    uno fuera del rango del deslizador, el deslizador amplía su rango. El valor que se usa es el de la casilla.
    tipo: "real", "entero" o "log" (deslizador logarítmico, para masas, longitudes o frecuencias).
    """

    def __init__(self, etiqueta, valor, unidad="", tipo="real", rango=(0.0, 1.0), paso=None, ayuda=""):
        self.etiqueta, self.tipo = etiqueta, tipo
        self._sincronizando = False
        disposicion = w.Layout(width="290px")
        if tipo == "entero":
            self.deslizador = w.IntSlider(value=int(valor), min=int(rango[0]), max=int(rango[1]), step=1,
                                          readout=False, continuous_update=False, layout=disposicion, style=_ESTILO)
        elif tipo == "log":
            self.deslizador = w.FloatLogSlider(value=float(valor), base=10, min=rango[0], max=rango[1],
                                               step=paso or 0.05, readout=False, continuous_update=False,
                                               layout=disposicion, style=_ESTILO)
        else:
            self.deslizador = w.FloatSlider(value=float(valor), min=rango[0], max=rango[1],
                                            step=paso or (rango[1] - rango[0]) / 200, readout=False,
                                            continuous_update=False, layout=disposicion, style=_ESTILO)
        self._rango_original = (self.deslizador.min, self.deslizador.max)
        self.texto = w.Text(value=self._formato(valor), continuous_update=False, layout=w.Layout(width="150px"))
        etiqueta_html = w.HTML(f"<div style='padding-top:4px' title='{_html.escape(ayuda)}'>{_html.escape(etiqueta)}</div>",
                               layout=w.Layout(width="250px"))
        unidad_html = w.HTML(f"<div style='padding-top:4px;opacity:.75'>{_html.escape(unidad)}</div>",
                             layout=w.Layout(width="200px"))
        self.caja = w.HBox([etiqueta_html, self.deslizador, self.texto, unidad_html],
                           layout=w.Layout(align_items="center", margin="1px 0"))
        self.deslizador.observe(self._desde_deslizador, "value")
        self.texto.observe(self._desde_texto, "value")

    def _formato(self, v):
        if self.tipo == "entero":
            return str(int(round(v)))
        return num(v, 6).replace("−", "-")

    def _desde_deslizador(self, cambio):
        if self._sincronizando:
            return
        self._sincronizando = True
        self.texto.value = self._formato(cambio["new"])
        self._sincronizando = False

    def _desde_texto(self, cambio):
        if self._sincronizando:
            return
        v, _ = leer_numero(cambio["new"])
        if v is None or not math.isfinite(v):
            return
        self._sincronizando = True
        try:
            d = self.deslizador
            lo, hi = self._rango_original
            # Si el valor vuelve a caber en el rango inicial, el deslizador recupera ese rango (más cómodo de usar)
            if self.tipo == "log" and v > 0 and lo <= math.log10(v) <= hi:
                d.min, d.max = lo, hi
            elif self.tipo != "log" and lo <= v <= hi:
                d.min, d.max = lo, hi
            if self.tipo == "log":
                if v > 0:
                    e = math.log10(v)
                    if e < d.min:
                        d.min = math.floor(e)
                    if e > d.max:
                        d.max = math.ceil(e)
                    d.value = v
            elif self.tipo == "entero":
                if float(v).is_integer() and abs(v) < 2 ** 31:
                    vi = int(v)
                    if vi < d.min:
                        d.min = vi
                    if vi > d.max:
                        d.max = vi
                    d.value = vi
            else:
                if v < d.min:
                    d.min = v
                if v > d.max:
                    d.max = v
                d.value = v
        except Exception:
            pass
        finally:
            self._sincronizando = False

    def leer(self, diag):
        """Valor numérico de la casilla; si no se puede leer, añade un ERROR al diagnóstico y devuelve None."""
        v, problema = leer_numero(self.texto.value)
        if problema:
            diag.error(f"{self.etiqueta}: {problema}.")
            return None
        if not math.isfinite(v):
            diag.error(f"{self.etiqueta} es infinito. Ninguna magnitud física medible es infinita; "
                       "escribe un valor finito, por grande que sea.")
            return None
        return v

    def fijar(self, v):
        self.texto.value = self._formato(v)

    def al_cambiar(self, funcion):
        self.texto.observe(lambda _: funcion(), "value")


def revisar_entero(v, etiqueta, diag, explicacion):
    """Comprueba que un número cuántico sea entero; si no lo es, explica por qué debe serlo."""
    if v is None:
        return None
    if abs(v - round(v)) > 1e-9 * max(1.0, abs(v)):
        diag.error(f"{etiqueta} = {num(v)} no es entero. {explicacion}")
        return None
    if abs(v) > 1e300:
        diag.error(f"{etiqueta} = {num(v)} es demasiado grande para representarlo (supera 1e300).")
        return None
    return int(round(v))


# ------------------------------------------------------------------
# Paneles de la interfaz
# ------------------------------------------------------------------
_BORDE = "1px solid rgba(128,128,128,0.45)"


def panel(titulo, filas):
    cabecera = w.HTML(f"<div style='font:600 11px monospace;letter-spacing:.14em;text-transform:uppercase;"
                      f"opacity:.7;padding-bottom:3px;border-bottom:{_BORDE};margin-bottom:4px'>"
                      f"{_html.escape(titulo)}</div>")
    return w.VBox([cabecera] + list(filas), layout=w.Layout(border=_BORDE, padding="8px 12px", margin="6px 0"))


def encabezado(titulo, subtitulo):
    display(HTML(f"<div style='margin:4px 0 10px 0'><div style='font:600 11px monospace;letter-spacing:.2em;"
                 f"opacity:.65'>QUANTUM LAB</div><div style='font-size:22px;font-weight:600;margin:2px 0'>"
                 f"{_html.escape(titulo)}</div><div style='opacity:.75'>{_html.escape(subtitulo)}</div></div>"))


def ayuda_entradas():
    return w.HTML("<div style='opacity:.7;font-size:12px;margin:2px 0 4px 0'>Escribe cualquier valor en las casillas "
                  "(admite 1e-30, 2,5, 3*10^8, 2*pi) y pulsa Enter, o usa el deslizador. Si un valor no tiene "
                  "sentido físico, el apartado «VALIDEZ FÍSICA DE LOS DATOS» explica qué ocurre.</div>")


def selector(etiqueta, opciones, valor=None, ancho="290px"):
    menu = w.Dropdown(options=opciones, value=valor, layout=w.Layout(width=ancho), style=_ESTILO)
    fila = w.HBox([w.HTML(f"<div style='padding-top:4px'>{_html.escape(etiqueta)}</div>", layout=w.Layout(width="250px")),
                   menu], layout=w.Layout(align_items="center", margin="1px 0"))
    return menu, fila


def casilla(texto, valor=False):
    return w.Checkbox(value=valor, description=texto, indent=False, layout=w.Layout(width="auto", margin="0 18px 0 0"))


def boton(texto, funcion, estilo="primary"):
    b = w.Button(description=texto, button_style=estilo, layout=w.Layout(width="auto", margin="4px 8px 4px 0"))
    b.on_click(lambda _: funcion())
    return b


def ejecutar_en(salida, funcion):
    """Ejecuta un cálculo dentro de una salida; cualquier fallo imprevisto se explica en lugar de romper el programa."""
    with salida:
        clear_output(wait=True)
        try:
            funcion()
        except Exception as error:      # noqa: BLE001 — se informa al usuario
            titulo_caja("NO SE PUDO COMPLETAR EL CÁLCULO")
            parrafo(f"Ocurrió un problema numérico inesperado: {type(error).__name__}: {error}")
            parrafo("Suele deberse a valores tan extremos que superan la precisión del ordenador. Prueba con "
                    "valores menos extremos; el resto del simulador sigue funcionando.")


# ------------------------------------------------------------------
# Estilo de las gráficas (fondo claro, como los simuladores de Bohr y del efecto fotoeléctrico)
# ------------------------------------------------------------------
COLOR = {"onda": "#2471a3", "densidad": "#d35400", "verde": "#147a4b", "rojo": "#c0392b", "morado": "#8e44ad",
         "naranja": "#f39c12", "gris": "#7f8c8d", "oscuro": "#34495e", "turquesa": "#16a085", "azul_claro": "#5dade2"}


def estilo_graficas():
    plt.rcdefaults()
    plt.rcParams.update({
        "figure.dpi": 100, "savefig.dpi": 110, "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
        "axes.labelsize": 11.5, "axes.grid": True, "grid.linestyle": "--", "grid.alpha": 0.35,
        "legend.fontsize": 9.5, "legend.framealpha": 0.95, "axes.spines.top": False, "axes.spines.right": False,
    })


def recuadro(ax, texto, x=0.97, y=0.95, color="#f39c12", ha="right", va="top", fondo="#fef9e7", tam=9.5):
    """Recuadro informativo como los de las gráficas de Bohr y del efecto fotoeléctrico."""
    ax.text(x, y, texto, transform=ax.transAxes, ha=ha, va=va, fontsize=tam, linespacing=1.5, zorder=10,
            bbox=dict(boxstyle="round,pad=0.55", facecolor=fondo, edgecolor=color, linewidth=1.6, alpha=0.96))


def mostrar(fig):
    """Muestra una figura de Matplotlib o de Plotly dentro de la salida de un widget."""
    if hasattr(fig, "savefig"):
        display(fig)
        plt.close(fig)
    else:
        pagina = fig.to_html(include_plotlyjs="cdn", full_html=True, config={"displaylogo": False, "responsive": True})
        alto = int(fig.layout.height or 600) + 20
        display(HTML(f'<iframe srcdoc="{_html.escape(pagina, quote=True)}" '
                     f'style="width:100%;height:{alto}px;border:0;background:white"></iframe>'))


estilo_graficas()
print("Utilidades de QUANTUM LAB listas.")


In [ ]:
#@title 2) Física del oscilador armónico cuántico 1D { display-mode: "form" }
# =====================================================================================
#  V(x) = ½·m·ω²·x²          x₀ = √(ħ/(mω))  (longitud característica)        ξ = x/x₀
#
#      ψₙ(x) = (1/√x₀)·φₙ(ξ),   φₙ(ξ) = Hₙ(ξ)·e^{−ξ²/2} / √(2ⁿ n! √π)          Eₙ = ħω(n + ½)
#
#  Las funciones φₙ se calculan con la recurrencia ESTABLE
#      φₖ₊₁ = √(2/(k+1))·ξ·φₖ − √(k/(k+1))·φₖ₋₁
#  con reescalado logarítmico: funciona para n de miles sin desbordar (los polinomios Hₙ directos
#  pierden toda la precisión hacia n ≈ 150).
#  Punto de retorno clásico: A = √(2n+1)·x₀ (donde Eₙ = V).
# =====================================================================================

N_NUMERICO_MAX = 3000       # hasta este n se calcula ψ en una malla; por encima, solo valores exactos


def longitud_caracteristica(m_kg, omega):
    return math.sqrt(HBAR / (m_kg * omega))


def funciones_hermite(n, xi):
    """Devuelve (φₙ₋₁, φₙ, φₙ₊₁) evaluadas en ξ con la recurrencia estable y reescalado."""
    xi = np.asarray(xi, dtype=float)
    base = -xi ** 2 / 2 - 0.25 * math.log(math.pi)
    anterior, actual = np.zeros_like(xi), np.ones_like(xi)
    escala = np.zeros_like(xi)
    guardadas = {}

    def valor(p):
        con_valor = p != 0
        out = np.zeros_like(p)
        out[con_valor] = np.sign(p[con_valor]) * np.exp(np.log(np.abs(p[con_valor])) + base[con_valor] + escala[con_valor])
        return out

    for k in range(0, n + 2):
        if k >= n - 1:
            guardadas[k] = valor(actual)
        siguiente = math.sqrt(2 / (k + 1)) * xi * actual - math.sqrt(k / (k + 1)) * anterior
        anterior, actual = actual, siguiente
        grande = np.abs(actual) > 1e150
        if grande.any():
            actual[grande] /= 1e150
            anterior[grande] /= 1e150
            escala[grande] += math.log(1e150)
    return guardadas.get(n - 1, np.zeros_like(xi)), guardadas[n], guardadas[n + 1]


def malla(n, rango=None, puntos=None):
    """Malla en ξ: cubre los puntos de retorno con holgura y resuelve las oscilaciones."""
    A = math.sqrt(2 * n + 1)
    semi = rango if rango is not None else A + 6.0
    if puntos is None:
        puntos = int(min(max(2001, 2 * semi * 20 * math.sqrt(2 * n + 1) / (2 * math.pi)), 120001))
    return np.linspace(-semi, semi, puntos)


def integrar(x, y):
    return float((getattr(np, "trapezoid", None) or np.trapz)(y, x))


def estado(n, rango=None, puntos=None):
    """Magnitudes del estado n en unidades naturales (ξ, ħω). Devuelve un diccionario."""
    xi = malla(n, rango, puntos)
    phi_ant, phi, phi_sig = funciones_hermite(n, xi)
    dphi = math.sqrt(n / 2) * phi_ant - math.sqrt((n + 1) / 2) * phi_sig          # derivada exacta
    rho = phi ** 2
    A = math.sqrt(2 * n + 1)
    u_T = 0.5 * dphi ** 2                    # densidad de energía cinética (en ħω/x₀)
    u_V = 0.5 * xi ** 2 * rho                # densidad de energía potencial
    T, V = integrar(xi, u_T), integrar(xi, u_V)
    # Probabilidad clásicamente prohibida: 1 − ∫ entre los puntos de retorno, en una malla que empieza y acaba
    # EXACTAMENTE en ±A (si no, el borde de la malla introduce un error del orden del paso).
    xi_dentro = np.linspace(-min(A, xi.max()), min(A, xi.max()), max(2001, len(xi) // 2))
    p_dentro = integrar(xi_dentro, funciones_hermite(n, xi_dentro)[1] ** 2)
    signos = np.sign(phi[np.abs(phi) > 1e-12 * np.abs(phi).max()])
    return {"xi": xi, "phi": phi, "dphi": dphi, "rho": rho, "A": A, "u_T": u_T, "u_V": u_V,
            "norma": integrar(xi, rho), "T": T, "V": V, "E_num": T + V,
            "x2": integrar(xi, xi ** 2 * rho), "P_fuera": max(integrar(xi, rho) - p_dentro, 0.0),
            "nodos": int(np.sum(signos[1:] != signos[:-1]))}


def densidad_clasica(xi, A):
    """Probabilidad clásica de encontrar la partícula en ξ (oscilador de amplitud A): 1/(π√(A² − ξ²))."""
    dentro = np.abs(xi) < A
    out = np.zeros_like(xi)
    out[dentro] = 1 / (math.pi * np.sqrt(A ** 2 - xi[dentro] ** 2))
    return out


def prob_fuera_n0():
    """P(|ξ| > 1) para n = 0: erfc(1) ≈ 0.1573 (exacta)."""
    return math.erfc(1.0)


# ------------------------------------------------------------------
# Validación con explicación física
# ------------------------------------------------------------------
EXPLICACION_N_OSC = ("Para que la función de onda se anule en ±∞ (partícula ligada), la serie de la ecuación de "
                     "Hermite tiene que cortarse en un polinomio, y eso solo ocurre si n es entero. Con n no entero ψ "
                     "crece como e^{+ξ²/2}: no es normalizable y no describe ninguna partícula.")


def revisar_n_osc(n_valor, diag, etiqueta="n"):
    n = revisar_entero(n_valor, etiqueta, diag, EXPLICACION_N_OSC)
    if n is None:
        return None
    if n < 0:
        diag.error(f"{etiqueta} = {n} es negativo. El estado de menor energía es n = 0, con E₀ = ħω/2 > 0: no existe "
                   f"ningún estado por debajo (su energía sería {n + 0.5}·ħω, menor que la energía mínima permitida por "
                   "el principio de incertidumbre). Matemáticamente, esas soluciones divergen en el infinito.")
        return None
    return n


def revisar_oscilador(m_me, omega, diag):
    m_kg = revisar_masa(m_me, diag)
    w_ok = None
    if omega is not None:
        if omega == 0:
            diag.error("ω = 0 significa que no hay fuerza recuperadora (resorte de constante k = mω² = 0): la "
                       "partícula es libre, no hay estados ligados ni niveles discretos y x₀ sería infinita.")
        elif omega < 0:
            diag.aviso(f"ω es negativa ({num(omega)} rad/s). El potencial ½mω²x² depende de ω², así que es el mismo "
                       f"oscilador que con ω = {num(-omega)} rad/s. Se usa el valor absoluto.")
            w_ok = -omega
        else:
            w_ok = omega
    if m_kg is None or w_ok is None:
        return None, None, None
    x0 = longitud_caracteristica(m_kg, w_ok)
    revisar_longitud(x0, diag, "La longitud característica x₀ = √(ħ/mω)")
    hw_ev = HBAR * w_ok / EV
    if hw_ev < 1e-3 * KT_AMBIENTE_EV:
        diag.nota(f"ħω = {num(hw_ev)} eV es mucho menor que la energía térmica a 300 K (0.026 eV): a temperatura "
                  "ambiente el oscilador estaría en un nivel altísimo y se comportaría de forma clásica.")
    if w_ok > 1e24:
        diag.aviso(f"ω = {num(w_ok)} rad/s corresponde a fotones de ħω = {num(hw_ev)} eV: rayos gamma de muy alta "
                   "energía. Ningún sistema ligado conocido vibra tan deprisa.")
    return m_kg, w_ok, x0


# ------------------------------------------------------------------
# Superposición: elementos de matriz exactos de x y x²
# ------------------------------------------------------------------
def matrices_x(ns):
    """⟨j|ξ|k⟩ y ⟨j|ξ²|k⟩ exactos (en unidades de x₀ y x₀²)."""
    J, K = np.meshgrid(ns, ns, indexing="ij")
    X = np.where(J == K - 1, np.sqrt(K / 2), 0.0) + np.where(J == K + 1, np.sqrt((K + 1) / 2), 0.0)
    X2 = (np.where(J == K, K + 0.5, 0.0) + np.where(J == K - 2, np.sqrt(np.maximum(K * (K - 1), 0)) / 2, 0.0)
          + np.where(J == K + 2, np.sqrt((K + 1) * (K + 2)) / 2, 0.0))
    return X, X2


def esperado(ns, cs, M, tau):
    b = cs * np.exp(-1j * (ns + 0.5) * tau)
    return float(np.real(np.conj(b) @ M @ b))


def onda_superposicion(xi, ns, cs, tau):
    psi = np.zeros_like(xi, dtype=complex)
    for n, c in zip(ns, cs):
        psi += c * np.exp(-1j * (n + 0.5) * tau) * funciones_hermite(int(n), xi)[1]
    return psi


In [ ]:
#@title 3) Gráficas del oscilador armónico { display-mode: "form" }
def figura_estado(n, d, x0, hw_ev):
    """ψₙ, |ψₙ|² con la densidad clásica, diagrama de potencial y niveles, densidades de energía."""
    f, u = escala_longitud(x0)
    s = x0 / f                                   # factor ξ → unidades de dibujo
    xi, A = d["xi"], d["A"]
    lim = min(A + 4.5, xi.max())
    sel = np.abs(xi) <= lim
    X = xi[sel] * s
    fig, ejes = plt.subplots(2, 2, figsize=(13.5, 9.6), constrained_layout=True)
    (ax1, ax2), (ax3, ax4) = ejes

    psi = d["phi"][sel] / math.sqrt(s)
    ax1.plot(X, psi, color=COLOR["onda"], lw=2.2, label=fr"$\psi_{{{n}}}(x)$")
    ax1.fill_between(X, psi, where=psi > 0, color=COLOR["onda"], alpha=0.15)
    ax1.fill_between(X, psi, where=psi < 0, color=COLOR["rojo"], alpha=0.12)
    for ax in (ax1, ax2):
        ax.axvspan(-lim * s, -A * s, color=COLOR["gris"], alpha=0.12)
        ax.axvspan(A * s, lim * s, color=COLOR["gris"], alpha=0.12, label="zona clásicamente prohibida")
        for sg in (-1, 1):
            ax.axvline(sg * A * s, color=COLOR["morado"], ls=":", lw=2)
    ax1.axhline(0, color="black", lw=0.8, alpha=0.7)
    ax1.set_xlim(-lim * s, lim * s)
    ax1.set_xlabel(f"posición x [{u}]")
    ax1.set_ylabel(f"ψ(x)  [{u}" + "$^{-1/2}$]")
    ax1.set_title(f"Función de onda (n = {n}, {n} nodos)")
    ax1.legend(loc="upper right", fontsize=8.5)

    rho = d["rho"][sel] / s
    clas = densidad_clasica(xi[sel], A) / s
    ax2.plot(X, rho, color=COLOR["densidad"], lw=2.3, label=r"$|\psi(x)|^2$ cuántica")
    ax2.fill_between(X, rho, color=COLOR["densidad"], alpha=0.18)
    ax2.plot(X, np.minimum(clas, 1.6 * rho.max()), color=COLOR["verde"], lw=1.8, ls="--", label="densidad clásica")
    ax2.set_xlim(-lim * s, lim * s)
    ax2.set_ylim(0, 1.35 * rho.max())
    ax2.set_xlabel(f"posición x [{u}]")
    ax2.set_ylabel(f"|ψ(x)|²  [{u}$^{{-1}}$]")
    ax2.set_title("Densidad de probabilidad frente a la clásica")
    ax2.legend(loc="upper right", fontsize=8.5)
    recuadro(ax2, f"Puntos de retorno ±A\nA = √(2n+1)·x₀ = {num(A * s, 4)} {u}\n"
                  f"P(zona prohibida) = {num(100 * d['P_fuera'], 4)} %", x=0.03, y=0.95, ha="left",
             color=COLOR["morado"])

    # Potencial y niveles
    xp = np.linspace(-lim, lim, 600)
    ax3.plot(xp * s, 0.5 * xp ** 2, color=COLOR["morado"], lw=2.4, label="V(x) = ½mω²x²")
    ks = range(max(0, n - 3), n + 4)
    for k in ks:
        Ak = math.sqrt(2 * k + 1)
        ax3.hlines(k + 0.5, -Ak * s, Ak * s, color=COLOR["rojo"] if k == n else COLOR["gris"],
                   lw=3 if k == n else 1.2, zorder=3)
        ax3.text(Ak * s + 0.03 * lim * s, k + 0.5, f"n = {k}", va="center", fontsize=9,
                 fontweight="bold" if k == n else "normal")
    esc = 0.8 / max(d["rho"].max(), 1e-300)
    ax3.plot(xi[sel] * s, n + 0.5 + d["rho"][sel] * esc, color=COLOR["densidad"], lw=1.6,
             label=r"$|\psi_n|^2$ sobre su nivel (escala arbitraria)")
    ax3.set_xlim(-lim * s, lim * s)
    ax3.set_ylim(max(0, n - 3.5), n + 4.6)
    ax3.set_xlabel(f"posición x [{u}]")
    ax3.set_ylabel("energía [ħω]")
    ax3.set_title("Potencial parabólico y niveles Eₙ = ħω(n + ½)")
    ax3.legend(loc="upper center", fontsize=8.5)
    recuadro(ax3, f"ħω = {num(hw_ev, 5)} eV\nEₙ = {num(hw_ev * (n + 0.5), 5)} eV\nniveles equiespaciados",
             x=0.03, y=0.95, ha="left", color=COLOR["rojo"])

    uT, uV = d["u_T"][sel] / s, d["u_V"][sel] / s
    ax4.plot(X, uT, color=COLOR["verde"], lw=2.2, label="cinética u_T = (ħ²/2m)|ψ'|²")
    ax4.plot(X, uV, color=COLOR["morado"], lw=2.2, ls="--", label="potencial u_V = V|ψ|²")
    if n == 0:
        recuadro(ax4, "En n = 0 las dos curvas coinciden\npunto a punto (u_T = u_V)", x=0.97, y=0.60, ha="right",
                 color=COLOR["morado"])
    ax4.fill_between(X, uT + uV, color=COLOR["naranja"], alpha=0.15, label="total")
    ax4.set_xlim(-lim * s, lim * s)
    ax4.set_xlabel(f"posición x [{u}]")
    ax4.set_ylabel(f"densidad de energía [ħω/{u}]")
    ax4.set_title("Dónde está la energía (teorema del virial)")
    ax4.legend(loc="upper right", fontsize=8.5)
    recuadro(ax4, f"⟨T⟩ = {num(d['T'], 5)} ħω\n⟨V⟩ = {num(d['V'], 5)} ħω\n⟨T⟩/⟨V⟩ = {num(d['T'] / d['V'], 5)}",
             x=0.03, y=0.95, ha="left", color=COLOR["verde"])
    fig.suptitle(f"QUANTUM LAB — Oscilador armónico cuántico 1D · estado n = {entero(n)}", fontsize=16, fontweight="bold")
    return fig


def figura_clasica(n, x0):
    """Para n enorme: densidad clásica (a la que tiende |ψ|²) y niveles."""
    f, u = escala_longitud(x0 * math.sqrt(2 * n + 1))
    A = math.sqrt(2 * n + 1) * x0 / f
    x = np.linspace(-A * 1.15, A * 1.15, 2001)
    fig, ax = plt.subplots(figsize=(12, 5.2), constrained_layout=True)
    dens = densidad_clasica(x, A)
    ax.plot(x, np.minimum(dens, 8 / (math.pi * A)), color=COLOR["verde"], lw=2.5, label="densidad clásica 1/(π√(A² − x²))")
    ax.fill_between(x, np.minimum(dens, 8 / (math.pi * A)), color=COLOR["verde"], alpha=0.15)
    for sg in (-1, 1):
        ax.axvline(sg * A, color=COLOR["morado"], ls=":", lw=2)
    ax.set_xlabel(f"posición x [{u}]")
    ax.set_ylabel(f"densidad [{u}$^{{-1}}$]")
    ax.set_title(f"Para n = {entero(n)}, |ψ|² oscila tan rápido que su promedio coincide con la densidad clásica")
    ax.legend(loc="upper center")
    recuadro(ax, f"amplitud clásica A = {num(A, 5)} {u}\nnodos: {entero(n)}\n(la partícula pasa más tiempo\n"
                 "cerca de los extremos, donde se frena)", x=0.5, y=0.62, ha="center", color=COLOR["morado"])
    fig.suptitle("QUANTUM LAB — Oscilador armónico · límite clásico", fontsize=16, fontweight="bold")
    return fig


def figura_superposicion(ns, cs, tau, x0, xi1, xi2):
    f, u = escala_longitud(x0)
    s = x0 / f
    A_max = math.sqrt(2 * ns.max() + 1)
    lim = A_max + 4
    xi = np.linspace(-lim, lim, int(min(max(1601, 40 * lim * math.sqrt(2 * ns.max() + 1)), 60001)))
    psi = onda_superposicion(xi, ns, cs, tau)
    rho = np.abs(psi) ** 2
    X, X2 = matrices_x(ns)
    taus = np.linspace(0, 2 * np.pi, 361)
    xm = np.array([esperado(ns, cs, X, t) for t in taus])
    dx = np.sqrt(np.maximum([esperado(ns, cs, X2, t) for t in taus] - xm ** 2, 0))
    fig = plt.figure(figsize=(13.5, 9.6), constrained_layout=True)
    gs = fig.add_gridspec(2, 2)
    ax1, ax2, ax3, ax4 = (fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[1, 0]),
                          fig.add_subplot(gs[1, 1]))
    ax1.plot(xi * s, psi.real / math.sqrt(s), color=COLOR["onda"], lw=2.1, label="Re Ψ")
    ax1.plot(xi * s, psi.imag / math.sqrt(s), color=COLOR["morado"], lw=1.9, ls="--", label="Im Ψ")
    ax1.axhline(0, color="black", lw=0.8, alpha=0.7)
    ax1.set_xlabel(f"posición x [{u}]")
    ax1.set_ylabel(f"Ψ(x, τ)  [{u}" + "$^{-1/2}$]")
    ax1.set_title(f"Función de onda en τ = ωt = {num(tau, 4)}")
    ax1.legend(loc="upper right")
    ax2.plot(xi * s, rho / s, color=COLOR["densidad"], lw=2.3, label="|Ψ(x, τ)|²")
    zona = (xi >= xi1) & (xi <= xi2)
    ax2.fill_between(xi * s, rho / s, where=zona, color=COLOR["verde"], alpha=0.25,
                     label=f"región [{num(xi1, 3)}, {num(xi2, 3)}]·x₀")
    m_actual = esperado(ns, cs, X, tau)
    ax2.axvline(m_actual * s, color=COLOR["rojo"], ls="--", lw=1.5, label="⟨x⟩")
    ax2.set_xlabel(f"posición x [{u}]")
    ax2.set_ylabel(f"|Ψ|²  [{u}$^{{-1}}$]")
    ax2.set_title("Densidad de probabilidad")
    ax2.legend(loc="upper right", fontsize=8.5)
    P = integrar(xi[zona], rho[zona]) if zona.sum() > 1 else 0.0
    recuadro(ax2, f"P(región) = {num(P, 5)}\n⟨x⟩ = {num(m_actual, 5)}·x₀", x=0.03, y=0.95, ha="left", color=COLOR["verde"])
    tm = np.linspace(0, 2 * np.pi, 181)
    xs_m = np.linspace(-lim, lim, 500)
    M = np.array([np.abs(onda_superposicion(xs_m, ns, cs, t)) ** 2 for t in tm])
    ax3.imshow(M, origin="lower", aspect="auto", extent=[-lim * s, lim * s, 0, 2 * np.pi], cmap="magma")
    ax3.plot(xm * s, taus, color="white", lw=1.4, ls="--", label="⟨x⟩(τ)")
    ax3.axhline(tau % (2 * math.pi), color=COLOR["naranja"], lw=1.3)
    ax3.set_xlabel(f"posición x [{u}]")
    ax3.set_ylabel("tiempo τ = ωt")
    ax3.set_title("Mapa espacio–tiempo de |Ψ|² (un periodo)")
    ax3.grid(False)
    ax3.legend(loc="upper right", fontsize=8.5)
    ax4.plot(taus, xm, color=COLOR["onda"], lw=2.3, label="⟨x⟩ / x₀")
    ax4.plot(taus, dx, color=COLOR["verde"], lw=2.3, label="Δx / x₀")
    ax4.axvline(tau % (2 * math.pi), color=COLOR["rojo"], ls="--", lw=1.4, label="τ elegido")
    ax4.set_xlim(0, 2 * np.pi)
    ax4.set_xlabel("tiempo τ = ωt (un periodo completo)")
    ax4.set_ylabel("[x₀]")
    ax4.set_title("Evolución de ⟨x⟩ y Δx (fórmulas exactas)")
    ax4.legend(loc="upper right")
    fig.suptitle("QUANTUM LAB — Oscilador armónico · superposición de estados", fontsize=16, fontweight="bold")
    return fig


In [ ]:
#@title 4) Resultados en texto (mismo formato que los simuladores de Bohr y del efecto fotoeléctrico) { display-mode: "form" }
def _revisar_malla(rango, puntos, n, diag):
    """Opciones numéricas: «auto» (None) o un valor; se explica cualquier valor problemático."""
    A = math.sqrt(2 * n + 1)
    if rango is not None:
        if rango <= 0:
            diag.aviso(f"El intervalo ±{num(rango)} x₀ no es válido (debe ser positivo): se usa el automático.")
            rango = None
        elif rango < A:
            diag.aviso(f"El intervalo ±{num(rango)} x₀ no llega a los puntos de retorno (±{num(A, 4)} x₀): se pierde "
                       "parte de la función de onda y la normalización numérica sale menor que 1.")
        elif rango > 50 * (A + 6):
            diag.nota(f"El intervalo ±{num(rango)} x₀ es enorme comparado con la zona donde está la partícula "
                      f"(±{num(A, 4)} x₀): casi todos los puntos caen donde ψ ≈ 0.")
    if puntos is not None:
        p = int(round(puntos))
        if p < 50:
            diag.aviso(f"{num(puntos)} puntos son demasiado pocos para representar ψ: se usan 50.")
            p = 50
        elif p > 200000:
            diag.nota(f"{num(puntos)} puntos harían el cálculo muy lento sin ganar precisión visible: se usan 200 000.")
            p = 200000
        semi = rango if rango is not None else A + 6
        necesarios = 2 * semi * 8 * A / (2 * math.pi)
        if p < necesarios:
            diag.aviso(f"Con {entero(p)} puntos no se resuelven las oscilaciones de ψ (hacen falta unos "
                       f"{entero(necesarios)}): las integrales numéricas pierden precisión.")
        puntos = p
    return rango, puntos


def informe_estado(n_valor, m_me, omega, rango=None, puntos=None, diag=None, dibujar=True):
    diag = diag or Diagnostico()
    titulo_caja("QUANTUM LAB — OSCILADOR ARMÓNICO CUÁNTICO 1D · ESTADO ESTACIONARIO")
    n = revisar_n_osc(n_valor, diag)
    m_kg, w_ok, x0 = revisar_oscilador(m_me, omega, diag)
    seccion("PARÁMETROS DEL SISTEMA")
    dato("Masa de la partícula m", num(m_me), "mₑ" + (f"   = {num(m_kg)} kg" if m_kg else ""))
    dato("Frecuencia angular ω", num(omega), "rad/s" + (f"   (f = ω/2π = {num(w_ok / (2 * math.pi))} Hz)" if w_ok else ""))
    dato("Número cuántico n", entero(n) if n is not None else num(n_valor))
    if diag.hay_error or n is None or m_kg is None:
        diag.imprimir()
        return None
    hw_ev = HBAR * w_ok / EV
    logE = math.log10(n + 0.5) + math.log10(HBAR * w_ok)
    revisar_relatividad(logE, m_kg, diag, f"La energía del nivel n = {entero(n)}")
    numerico = n <= N_NUMERICO_MAX
    if not numerico:
        diag.nota(f"Con n = {entero(n)} la función de onda tiene {entero(n)} nodos; representarla exigiría millones de "
                  "puntos. Se dan los valores exactos (fórmulas) y la densidad clásica, a la que tiende |ψ|² cuando n "
                  "es grande (principio de correspondencia).")
    elif n > 100:
        diag.nota(f"Con n = {entero(n)} la densidad cuántica, promediada sobre sus oscilaciones, ya es casi idéntica a "
                  "la clásica (principio de correspondencia).")
    if numerico:
        rango, puntos = _revisar_malla(rango, puntos, n, diag)
    diag.imprimir()

    A = math.sqrt(2 * n + 1)
    seccion("RESULTADOS DEL ESTADO")
    dato("Cuanto de energía ħω", num(hw_ev), "eV" + f"   = {num(HBAR * w_ok)} J")
    dato("Energía Eₙ = ħω(n + ½)", num_log(logE - math.log10(EV)), "eV")
    dato("Energía Eₙ", num_log(logE), "J")
    dato("Energía del fundamental E₀ = ħω/2", num(hw_ev / 2), "eV  (energía de punto cero)")
    dato("Separación entre niveles", num(hw_ev), "eV  (todos equiespaciados)")
    dato("Longitud característica x₀ = √(ħ/mω)", num(x0), "m" + f"   = {num(x0 * 1e9)} nm")
    dato("Punto de retorno clásico A = √(2n+1)·x₀", num(A * x0), "m")
    dato("Constante del resorte k = mω²", num(m_kg * w_ok ** 2), "N/m")
    dato("⟨x²⟩ exacto = (n + ½)·x₀²", num((n + 0.5), 8), "· x₀²")
    if numerico:
        d = estado(n, rango, puntos)
        dato("⟨x²⟩ numérico", num(d["x2"], 8), "· x₀²")
        dato("Normalización numérica ∫|ψ|²dx", f"{d['norma']:.8f}", "(debe ser 1)")
        dato("Energía numérica ∫(u_T + u_V)dx", num(d["E_num"], 8), f"· ħω   (exacta: {n + 0.5})")
        err = abs(d["E_num"] / (n + 0.5) - 1)
        dato("Error relativo de la energía", num(err, 3) if err > 1e-15 else "< 1e−15 (precisión de la máquina)")
        dato("Virial: ⟨T⟩/E y ⟨V⟩/E", f"{d['T'] / d['E_num']:.6f}  y  {d['V'] / d['E_num']:.6f}", "(ambos ½)")
        dato("Nodos encontrados", f"{d['nodos']}", f"(esperados: {n})")
        dato("Probabilidad en la zona prohibida", f"{100 * d['P_fuera']:.5f}", "%")
        dato("Puntos de la malla / intervalo", f"{entero(len(d['xi']))}  /  ±{num(d['xi'].max(), 4)}", "x₀")
    seccion("NIVELES DE ENERGÍA CERCANOS")
    tabla(["n", "Eₙ [eV]", "Eₙ / ħω", "amplitud clásica A [x₀]"],
          [[("» " if k == n else "") + entero(k), num_log(math.log10(k + 0.5) + math.log10(hw_ev)), num(k + 0.5),
            num(math.sqrt(2 * k + 1), 6)] for k in range(max(0, n - 2), n + 3)])
    seccion("INTERPRETACIÓN FÍSICA")
    vinetas([
        "Los niveles están igualmente espaciados (ΔE = ħω): por eso las moléculas absorben luz infrarroja en "
        "frecuencias muy concretas.",
        "E₀ = ħω/2 ≠ 0: incluso en el estado fundamental la partícula vibra (energía de punto cero).",
        f"La función de onda tiene n = {entero(n)} nodos y se extiende un poco más allá de los puntos de retorno: hay "
        "probabilidad de encontrar la partícula donde la física clásica lo prohíbe (efecto túnel).",
        "En promedio la energía se reparte a partes iguales entre cinética y potencial (teorema del virial).",
        "Para n grande |ψ|² se parece a la densidad clásica: la partícula pasa más tiempo cerca de los extremos.",
    ])
    if dibujar:
        mostrar(figura_estado(n, d, x0, hw_ev) if numerico else figura_clasica(n, x0))
    return {"n": n, "E_ev": hw_ev * (n + 0.5), "numerico": numerico}


def informe_superposicion(estados, tau, xi1, xi2, m_me, omega, diag=None, dibujar=True):
    diag = diag or Diagnostico()
    titulo_caja("QUANTUM LAB — OSCILADOR ARMÓNICO · SUPERPOSICIÓN DE ESTADOS")
    m_kg, w_ok, x0 = revisar_oscilador(m_me, omega, diag)
    if not estados:
        diag.error("No hay ningún estado activo. Marca al menos una casilla «activo».")
    combinados = {}
    for i, (n_v, A, fase) in enumerate(estados, 1):
        n = revisar_n_osc(n_v, diag, f"n del estado {i}")
        if n is None or A is None or fase is None:
            continue
        if n > N_NUMERICO_MAX:
            diag.error(f"n del estado {i} = {entero(n)}: la superposición se dibuja en una malla y solo admite n ≤ "
                       f"{entero(N_NUMERICO_MAX)}. (En el estado individual sí se aceptan n mayores.)")
            continue
        if A == 0:
            diag.nota(f"La amplitud del estado {i} es 0: ese estado no contribuye.")
            continue
        if A < 0:
            diag.nota(f"La amplitud del estado {i} es negativa: equivale a amplitud {num(-A)} con la fase aumentada en π.")
        if abs(fase) > 2 * math.pi:
            diag.nota(f"La fase del estado {i} ({num(fase)} rad) supera 2π: equivale a {num(math.remainder(fase, 2 * math.pi))} rad.")
        if n in combinados:
            diag.nota(f"El nivel n = {n} aparece más de una vez: sus coeficientes se suman.")
        combinados[n] = combinados.get(n, 0) + A * complex(math.cos(fase), math.sin(fase))
    combinados = {n: c for n, c in combinados.items() if abs(c) > 0}
    if not diag.hay_error and not combinados:
        diag.error("Todas las amplitudes son cero: la función de onda sería 0 y no habría partícula.")
    if tau is not None and abs(tau) > 2 * math.pi:
        diag.nota(f"τ = {num(tau)} supera un periodo: |Ψ|² se repite exactamente cada Δτ = 2π. Se usa τ = "
                  f"{num(tau % (2 * math.pi))}.")
    if xi1 is not None and xi2 is not None and xi1 > xi2:
        diag.nota("El inicio de la región es mayor que el final: se intercambian.")
        xi1, xi2 = xi2, xi1
    seccion("PARÁMETROS DEL SISTEMA")
    dato("Masa de la partícula m", num(m_me), "mₑ")
    dato("Frecuencia angular ω", num(omega), "rad/s")
    dato("Tiempo τ = ωt", num(tau))
    dato("Región estudiada", f"[{num(xi1)}, {num(xi2)}]·x₀")
    if diag.hay_error or m_kg is None or tau is None or xi1 is None or xi2 is None:
        diag.imprimir()
        return None
    diag.imprimir()
    ns = np.array(sorted(combinados), dtype=float)
    cs = np.array([combinados[int(n)] for n in ns], dtype=complex)
    cs = cs / math.sqrt(float(np.sum(np.abs(cs) ** 2)))
    tau_r = tau % (2 * math.pi)
    X, X2 = matrices_x(ns)
    hw_ev = HBAR * w_ok / EV
    seccion("COEFICIENTES NORMALIZADOS")
    tabla(["n", "cₙ (normalizado)", "|cₙ|²", "Eₙ [eV]"],
          [[entero(n), f"{c.real:+.4f} {c.imag:+.4f}i", f"{abs(c) ** 2:.6f}", num(hw_ev * (n + 0.5))] for n, c in zip(ns, cs)])
    xm = esperado(ns, cs, X, tau_r)
    dx = math.sqrt(max(esperado(ns, cs, X2, tau_r) - xm ** 2, 0))
    pesos = np.abs(cs) ** 2
    e_med = float(np.sum(pesos * (ns + 0.5)))
    seccion("RESULTADOS EN EL INSTANTE ELEGIDO")
    dato("Tiempo real t = τ/ω", num(tau / w_ok), "s")
    dato("Periodo clásico T = 2π/ω", num(2 * math.pi / w_ok), "s")
    dato("Posición media ⟨x⟩", num(xm, 6), f"· x₀   = {num(xm * x0)} m")
    dato("Incertidumbre Δx", num(dx, 6), "· x₀")
    dato("Energía media ⟨E⟩", num(e_med * hw_ev), "eV")
    dato("Dispersión ΔE", num(math.sqrt(max(float(np.sum(pesos * (ns + 0.5) ** 2)) - e_med ** 2, 0)) * hw_ev), "eV")
    seccion("INTERPRETACIÓN FÍSICA")
    lineas = ["⟨x⟩ oscila exactamente con la frecuencia clásica ω cuando hay estados con n consecutivos (Δn = ±1): "
              "es la conexión con el muelle clásico (teorema de Ehrenfest).",
              "Si los n difieren en un número par, ⟨x⟩ = 0 siempre, pero la anchura Δx «respira» con frecuencia 2ω.",
              "|cₙ|² no cambia con el tiempo: la energía media y su dispersión son constantes."]
    if len(ns) == 1:
        lineas.insert(0, "Solo hay un estado activo: es estacionario y |Ψ|² no cambia.")
    vinetas(lineas)
    if dibujar:
        mostrar(figura_superposicion(ns, cs, tau_r, x0, xi1, xi2))
    return {"x": xm, "dx": dx}


In [ ]:
#@title 5) Pruebas numéricas (comprueban que los cálculos son correctos) { display-mode: "form" }
def ejecutar_pruebas():
    import io, contextlib
    from scipy.special import eval_hermite, factorial
    resultados = []

    def prueba(nombre, condicion, detalle=""):
        resultados.append((nombre, bool(condicion), detalle))

    def silencioso(funcion, *args, **kwargs):
        with contextlib.redirect_stdout(io.StringIO()):
            return funcion(*args, **kwargs)

    xi = np.linspace(-8, 8, 4001)
    for n in (0, 3, 10):
        directa = eval_hermite(n, xi) * np.exp(-xi ** 2 / 2) / math.sqrt(2 ** n * factorial(n) * math.sqrt(math.pi))
        prueba(f"Recurrencia estable = fórmula con Hₙ (n = {n})", np.max(np.abs(funciones_hermite(n, xi)[1] - directa)) < 1e-12)
    for n in (0, 5, 50, 1500):
        d = estado(n)
        prueba(f"Normalización ∫|ψ{n}|² = 1", abs(d["norma"] - 1) < 1e-6, f"{d['norma']:.9f}")
        prueba(f"Energía numérica = ħω(n + ½) (n = {n})", abs(d["E_num"] / (n + 0.5) - 1) < 1e-5, f"{d['E_num']:.6f}")
        prueba(f"Virial ⟨T⟩ = ⟨V⟩ (n = {n})", abs(d["T"] / d["V"] - 1) < 1e-5)
        prueba(f"Número de nodos = n (n = {n})", d["nodos"] == n, f"{d['nodos']}")
    d = estado(0)
    prueba("P(zona prohibida) del fundamental = erfc(1) ≈ 15.73 %", abs(d["P_fuera"] - prob_fuera_n0()) < 1e-5,
           f"{100 * d['P_fuera']:.4f} %")
    d4 = estado(4)
    prueba("⟨x²⟩ = (n + ½)x₀² (n = 4)", abs(d4["x2"] - 4.5) < 1e-6)
    xi2 = d4["xi"]
    prueba("Derivada exacta = derivada numérica", np.max(np.abs(np.gradient(d4["phi"], xi2) - d4["dphi"])) < 2e-4)
    _, f0, _ = funciones_hermite(0, xi)
    _, f1, _ = funciones_hermite(1, xi)
    prueba("Ortogonalidad ∫ψ₀ψ₁ = 0", abs(integrar(xi, f0 * f1)) < 1e-10)
    ns, cs = np.array([0.0, 1.0, 2.0]), np.array([0.6, 0.64j, 0.48])
    cs = cs / np.sqrt(np.sum(np.abs(cs) ** 2))
    X, X2 = matrices_x(ns)
    for tau in (0.0, 1.1):
        rho = np.abs(onda_superposicion(xi, ns, cs, tau)) ** 2
        prueba(f"⟨x⟩ exacto = numérico (τ = {tau})", abs(integrar(xi, xi * rho) - esperado(ns, cs, X, tau)) < 1e-8)
        prueba(f"⟨x²⟩ exacto = numérico (τ = {tau})", abs(integrar(xi, xi ** 2 * rho) - esperado(ns, cs, X2, tau)) < 1e-8)
    prueba("Periodicidad de |Ψ|² con periodo 2π", np.allclose(np.abs(onda_superposicion(xi, ns, cs, 0.4)) ** 2,
                                                         np.abs(onda_superposicion(xi, ns, cs, 0.4 + 2 * math.pi)) ** 2))
    hw = HBAR * 1e15 / EV
    prueba("Electrón con ω = 1e15 rad/s: ħω ≈ 0.6582 eV", abs(hw - 0.658212) < 1e-5, f"{hw:.6f} eV")
    for etiqueta, args in [("n = 2.5", (2.5, 1, 1e15)), ("n = −1", (-1, 1, 1e15)), ("ω = 0", (0, 1, 0)),
                           ("masa 0", (0, 0, 1e15))]:
        dd = Diagnostico()
        r = silencioso(informe_estado, *args, diag=dd, dibujar=False)
        prueba(f"Dato no físico ({etiqueta}): se explica y no se calcula", r is None and dd.hay_error)
    dd = Diagnostico()
    r = silencioso(informe_estado, 1, 1, -1e15, diag=dd, dibujar=False)
    prueba("ω negativa se trata como |ω|", r is not None and abs(r["E_ev"] - 1.5 * hw) < 1e-9)
    dd = Diagnostico()
    r = silencioso(informe_estado, 10 ** 9, 1, 1e15, diag=dd, dibujar=False)
    prueba("n = 10⁹: valores exactos sin desbordar y aviso relativista", r is not None and not r["numerico"]
           and any(n_ == "AVISO" for n_, _ in dd.items))

    titulo_caja("PRUEBAS NUMÉRICAS — OSCILADOR ARMÓNICO")
    tabla(["#", "Comprobación", "Resultado", "Detalle"], alinear="clcl", sangria=1,
          filas=[[i, nombre, "Correcta" if ok else "FALLA", detalle] for i, (nombre, ok, detalle) in enumerate(resultados, 1)])
    correctas = sum(ok for _, ok, _ in resultados)
    print(f"\n   {correctas} de {len(resultados)} pruebas superadas.")
    return correctas, len(resultados)


In [ ]:
#@title 6) Simulador interactivo — oscilador armónico cuántico 1D { display-mode: "form" }
EJEMPLOS = {
    "Electrón, ω = 1e15 rad/s (n = 0)": {"m": "1", "w": "1e15", "n": "0"},
    "Electrón, ω = 1e15 rad/s (n = 2)": {"m": "1", "w": "1e15", "n": "2"},
    "Vibración de la molécula H₂ (μ ≈ 918 mₑ, ω ≈ 8.3e14 rad/s)": {"m": "918", "w": "8.29e14", "n": "0"},
    "Límite clásico: n = 60": {"m": "1", "w": "1e15", "n": "60"},
    "Valor extremo: n = 10^9": {"m": "1", "w": "1e15", "n": "1e9"},
    "Valor extremo: ω = 1e30 rad/s": {"m": "1", "w": "1e30", "n": "0"},
    "Valor no físico: n = −1": {"m": "1", "w": "1e15", "n": "-1"},
    "Valor no físico: ω = 0": {"m": "1", "w": "0", "n": "0"},
}

entrada_m = Entrada("Masa de la partícula m", 1.0, "mₑ (masas del electrón)", tipo="log", rango=(-1, 4))
entrada_w = Entrada("Frecuencia angular ω", 1e15, "rad/s", tipo="log", rango=(12, 17))
entrada_n = Entrada("Número cuántico n", 0, "", tipo="entero", rango=(0, 30))
entrada_rango = w.Text(value="auto", continuous_update=False, layout=w.Layout(width="150px"))
entrada_puntos = w.Text(value="auto", continuous_update=False, layout=w.Layout(width="150px"))
fila_numerica = w.HBox([w.HTML("<div style='padding-top:4px'>Intervalo ± [x₀] y puntos de la malla</div>",
                               layout=w.Layout(width="250px")), entrada_rango, entrada_puntos,
                        w.HTML("<div style='padding-top:4px;opacity:.7'>&nbsp;«auto» = elección automática</div>")],
                       layout=w.Layout(align_items="center"))
menu_ejemplo, fila_ejemplo = selector("Configuración de ejemplo", ["— elegir —"] + list(EJEMPLOS), "— elegir —", "420px")

_cab = w.HBox([w.HTML("<b>Estado</b>", layout=w.Layout(width="80px")), w.HTML("<b>activo</b>", layout=w.Layout(width="70px")),
               w.HTML("<b>n</b>", layout=w.Layout(width="150px")), w.HTML("<b>amplitud A</b>", layout=w.Layout(width="150px")),
               w.HTML("<b>fase φ [rad]</b>", layout=w.Layout(width="150px"))])
filas_sup = []
for i, (n0, A0, f0, act) in enumerate([("0", "1", "0", True), ("1", "1", "0", True), ("2", "1", "0", False)], 1):
    filas_sup.append((w.Checkbox(value=act, indent=False, layout=w.Layout(width="70px")),
                      w.Text(value=n0, continuous_update=False, layout=w.Layout(width="140px")),
                      w.Text(value=A0, continuous_update=False, layout=w.Layout(width="140px")),
                      w.Text(value=f0, continuous_update=False, layout=w.Layout(width="140px"))))
caja_estados = w.VBox([_cab] + [w.HBox([w.HTML(f"{i}", layout=w.Layout(width="80px")), act, n, A, fa])
                                for i, (act, n, A, fa) in enumerate(filas_sup, 1)])
entrada_tau = Entrada("Tiempo τ = ωt", 0.0, "(periodo 2π)", tipo="real", rango=(0, 2 * math.pi), paso=0.05)
entrada_x1 = Entrada("Región: inicio", 0.0, "· x₀", tipo="real", rango=(-5, 5), paso=0.05)
entrada_x2 = Entrada("Región: final", 5.0, "· x₀", tipo="real", rango=(-5, 5), paso=0.05)

salida_estado, salida_sup, salida_pruebas = w.Output(), w.Output(), w.Output()


def _leer_texto(caja_texto, etiqueta, diag, auto=False):
    if auto and caja_texto.value.strip().lower() in ("auto", ""):
        return None
    v, problema = leer_numero(caja_texto.value)
    if problema:
        diag.error(f"{etiqueta}: {problema}.")
        return None
    if not math.isfinite(v):
        diag.error(f"{etiqueta} es infinito: escribe un valor finito.")
        return None
    return v


def calcular_estado():
    def tarea():
        d = Diagnostico()
        informe_estado(entrada_n.leer(d), entrada_m.leer(d), entrada_w.leer(d),
                       _leer_texto(entrada_rango, "Intervalo", d, auto=True),
                       _leer_texto(entrada_puntos, "Puntos de la malla", d, auto=True), d)
    ejecutar_en(salida_estado, tarea)


def calcular_superposicion():
    def tarea():
        d = Diagnostico()
        estados = [(_leer_texto(n, f"n del estado {i}", d), _leer_texto(A, f"Amplitud del estado {i}", d),
                    _leer_texto(fa, f"Fase del estado {i}", d))
                   for i, (act, n, A, fa) in enumerate(filas_sup, 1) if act.value]
        informe_superposicion(estados, entrada_tau.leer(d), entrada_x1.leer(d), entrada_x2.leer(d),
                              entrada_m.leer(d), entrada_w.leer(d), d)
    ejecutar_en(salida_sup, tarea)


def calcular_todo():
    calcular_estado()
    calcular_superposicion()


def _aplicar_ejemplo(cambio):
    e = EJEMPLOS.get(cambio["new"])
    if e:
        entrada_m.texto.value, entrada_w.texto.value, entrada_n.texto.value = e["m"], e["w"], e["n"]


menu_ejemplo.observe(_aplicar_ejemplo, "value")
for e in (entrada_m, entrada_w):
    e.al_cambiar(calcular_todo)
entrada_n.al_cambiar(calcular_estado)
for c in (entrada_rango, entrada_puntos):
    c.observe(lambda _: calcular_estado(), "value")
for e in (entrada_tau, entrada_x1, entrada_x2):
    e.al_cambiar(calcular_superposicion)
for fila in filas_sup:
    for control in fila:
        control.observe(lambda _: calcular_superposicion(), "value")

pestanas = w.Tab(children=[
    w.VBox([panel("Estado estacionario", [entrada_n.caja, fila_numerica]), salida_estado]),
    w.VBox([panel("Estados de la superposición (se normalizan automáticamente)", [caja_estados]),
            panel("Tiempo y región de probabilidad", [entrada_tau.caja, entrada_x1.caja, entrada_x2.caja]),
            salida_sup]),
    w.VBox([boton("Ejecutar las pruebas numéricas", lambda: ejecutar_en(salida_pruebas, ejecutar_pruebas)), salida_pruebas]),
])
for i, t in enumerate(["Estado individual", "Superposición", "Pruebas numéricas"]):
    pestanas.set_title(i, t)

encabezado("Oscilador armónico cuántico 1D",
           "Potencial V = ½mω²x²: polinomios de Hermite, niveles equiespaciados, energía de punto cero, zona "
           "clásicamente prohibida, teorema del virial, límite clásico y dinámica de superposiciones.")
display(w.VBox([panel("Parámetros del sistema", [ayuda_entradas(), fila_ejemplo, entrada_m.caja, entrada_w.caja]),
                pestanas]))
calcular_todo()
